# Kathaa Studio — free-GPU worker

Runs a batch of image and video jobs exported from Kathaa on this notebook's free GPU, then saves the files and a `results.json` manifest to the Output tab.

**Settings (right panel):** Accelerator → *GPU T4 x2*, Internet → *On*.

**Each batch:** paste Kathaa's job list into the first code cell, then *Save Version → Save & Run All* so it runs in the background. Do not use *Run All* in the editor for real batches: an editor session stops after about 40 minutes without edits and its files are lost. When it finishes, download the output zip and import it into Kathaa (Vault → Import results).

Models (all Apache-2.0, commercial use allowed): Wan 2.2 TI2V-5B and Wan 2.1 T2V-1.3B for video, Z-Image-Turbo and FLUX.1-schnell for images. Batch only — no tunnels or remote UIs.

In [ ]:
# 1 — Paste the job list exported from Kathaa between the triple quotes.
JOBS = r'''
{
 "app": "kathaa",
 "type": "jobs",
 "version": 1,
 "jobs": [
  {
   "id": "smoke_img",
   "engine": "z-image-turbo",
   "kind": "image",
   "title": "Smoke test image",
   "prompt": "Golden-hour photograph of the excavated mud-brick streets of Mohenjo-daro, long shadows, archaeological site, realistic, 35mm",
   "width": 1280,
   "height": 720,
   "seed": 7
  },
  {
   "id": "smoke_vid_wan22_480p",
   "engine": "wan2.2-ti2v-5b",
   "kind": "video",
   "title": "Smoke test video (Wan 2.2, 480p)",
   "prompt": "Slow cinematic dolly forward along an excavated mud-brick street of the Indus Valley city of Mohenjo-daro at sunrise, low brick walls of fired bricks, dust in warm light, realistic documentary footage",
   "width": 832,
   "height": 480,
   "seconds": 2,
   "seed": 7,
   "steps": 30
  }
 ]
}
'''


In [ ]:
# 2 — Install the current model libraries (needs Internet: On)
!pip install -q -U diffusers transformers accelerate ftfy sentencepiece imageio imageio-ffmpeg 2>&1 | tail -n 3
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # set before CUDA starts
import torch, diffusers
print('torch', torch.__version__, '| diffusers', diffusers.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 3 — Worker code
"""Kathaa free-GPU worker.

Runs a batch of generation jobs exported from Kathaa Studio on a notebook GPU
(built for Kaggle's free T4 GPUs) and writes the results, plus a results.json
manifest, to OUT_DIR. Kathaa imports that folder (or its zip) into the Vault.

Batch only: nothing here opens a port, tunnel or remote UI.
All models are Apache-2.0 licensed, so outputs can be used commercially.
"""

from __future__ import annotations

import base64
import gc
import io
import json
import math
import os
import random
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone

OUT_DIR = os.environ.get("KATHAA_OUT", "/kaggle/working/kathaa-out")
# Lets PyTorch reuse fragmented GPU memory; must be set before torch is imported.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

ENGINES = {
    # video
    "wan2.2-ti2v-5b": {"kind": "video", "repo": "Wan-AI/Wan2.2-TI2V-5B-Diffusers", "fps": 24, "mult": 32},
    "wan2.1-t2v-1.3b": {"kind": "video", "repo": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", "fps": 16, "mult": 16},
    # image
    # float16 gives a blank Z-Image on a T4 (measured 23 Nov 2026), so it goes straight to bfloat16
    "z-image-turbo": {"kind": "image", "repo": "Tongyi-MAI/Z-Image-Turbo", "mult": 16, "no_fp16": True},
    "flux-schnell": {"kind": "image", "repo": "black-forest-labs/FLUX.1-schnell", "mult": 16},
}

NEGATIVE_VIDEO = (
    "blurry, low quality, distorted, deformed, extra limbs, watermark, text, subtitles, "
    "static, frozen frame, flicker, jpeg artefacts"
)


def free_disk_gb(path: str = "/") -> float:
    import shutil

    return round(shutil.disk_usage(path).free / 1024**3, 1)


def purge_model_cache(repo: str) -> None:
    """Kaggle sessions have about 57 GB of disk: drop a model's download once its jobs are done."""
    try:
        from huggingface_hub import scan_cache_dir

        cache = scan_cache_dir()
        hashes = [rev.commit_hash for r in cache.repos if r.repo_id == repo for rev in r.revisions]
        if hashes:
            strategy = cache.delete_revisions(*hashes)
            log(f"Freed {strategy.expected_freed_size_str} by removing {repo} from the cache")
            strategy.execute()
    except Exception as e:  # noqa: BLE001
        log(f"Could not clear {repo} from the cache: {e}")


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def log(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def gpu_summary() -> dict:
    import torch

    info = {"cuda": torch.cuda.is_available(), "gpus": [], "torch": torch.__version__}
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            info["gpus"].append({"name": p.name, "vram_gb": round(p.total_memory / 1024**3, 1), "cc": f"{p.major}.{p.minor}"})
    try:
        import diffusers

        info["diffusers"] = diffusers.__version__
    except Exception:  # noqa: BLE001
        info["diffusers"] = None
    return info


def snap(value: int, mult: int, lo: int, hi: int) -> int:
    value = max(lo, min(hi, int(value)))
    return max(mult, int(round(value / mult)) * mult)


def decode_image(data_url: str):
    from PIL import Image

    b64 = data_url.split(",", 1)[1] if "," in data_url else data_url
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")


def fit_cover(img, width: int, height: int):
    """Scale and centre-crop so the start frame fills the target size."""
    from PIL import Image

    scale = max(width / img.width, height / img.height)
    w, h = math.ceil(img.width * scale), math.ceil(img.height * scale)
    img = img.resize((w, h), Image.LANCZOS)
    left, top = (w - width) // 2, (h - height) // 2
    return img.crop((left, top, left + width, top + height))


def frames_look_broken(frames) -> bool:
    """True when the output is all black / NaN (typical half-precision failure on older GPUs)."""
    import numpy as np

    arr = np.asarray(frames[0] if isinstance(frames, (list, tuple)) else frames, dtype=np.float32)
    if not np.isfinite(arr).all():
        return True
    if arr.max() <= 1.5:  # 0..1 range
        arr = arr * 255
    return float(arr.mean()) < 2.0 or float(arr.std()) < 1.0


class Worker:
    def __init__(self) -> None:
        import torch

        self.torch = torch
        self.pipe = None
        self.pipe_key = None
        self.te = None
        self.split = False
        # T4 (compute 7.5) has no native bfloat16; try float16 first, fall back to bfloat16 if output breaks.
        cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
        self.dtypes = [torch.bfloat16] if cc[0] >= 8 else [torch.float16, torch.bfloat16]

    # ---------- pipeline loading ----------
    def unload(self) -> None:
        """Free every GPU byte held by the previous model before loading another one."""
        if self.pipe is not None:
            try:
                self.pipe.remove_all_hooks()  # CPU-offload hooks keep references to the weights
            except Exception:  # noqa: BLE001
                pass
        self.pipe = None
        self.pipe_key = None
        self.te = None
        self.split = False
        gc.collect()
        if self.torch.cuda.is_available():
            self.torch.cuda.synchronize()
            self.torch.cuda.empty_cache()
            self.torch.cuda.ipc_collect()

    def dtypes_for(self, engine: str) -> list:
        if ENGINES[engine].get("no_fp16"):
            return [d for d in self.dtypes if d != self.torch.float16] or [self.torch.bfloat16]
        return self.dtypes

    def load(self, engine: str, mode: str, dtype) -> None:
        key = (engine, mode, str(dtype))
        if self.pipe_key == key:
            return
        self.unload()
        torch = self.torch
        repo = ENGINES[engine]["repo"]
        log(f"Loading {engine} ({mode}, {dtype}) from {repo} …")
        t0 = time.time()
        if engine.startswith("wan"):
            from diffusers import AutoencoderKLWan

            vae = AutoencoderKLWan.from_pretrained(repo, subfolder="vae", torch_dtype=torch.float32)
            if mode == "i2v":
                from diffusers import WanImageToVideoPipeline as P
            else:
                from diffusers import WanPipeline as P
            if torch.cuda.device_count() >= 2:
                # Text encoder (about 11 GB) on the second T4, transformer and VAE on the first:
                # keeps system RAM low (offloading everything to RAM crashed the 30 GB session at 720p)
                # and uses both free GPUs.
                from transformers import UMT5EncoderModel

                self.te = UMT5EncoderModel.from_pretrained(repo, subfolder="text_encoder", torch_dtype=dtype).to("cuda:1")
                pipe = P.from_pretrained(repo, vae=vae, text_encoder=self.te, torch_dtype=dtype)
                pipe.text_encoder = None  # so the pipeline runs on cuda:0; prompts are encoded separately
                pipe.transformer.to("cuda:0")
                pipe.vae.to("cuda:0")
                self.split = True
            else:
                pipe = P.from_pretrained(repo, vae=vae, torch_dtype=dtype)
                pipe.enable_model_cpu_offload()
                self.split = False
            try:
                pipe.vae.enable_tiling()
            except Exception:  # noqa: BLE001
                pass
        elif engine == "z-image-turbo":
            from diffusers import ZImagePipeline

            pipe = ZImagePipeline.from_pretrained(repo, torch_dtype=dtype)
            if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
                # On a T4, bfloat16 attention has no memory-efficient kernel, so the whole 6B transformer
                # plus 720p attention does not fit in 15 GB (out of memory, 23 Nov 2026). Stream the
                # layers through the GPU one at a time instead: slower, but it fits.
                pipe.enable_sequential_cpu_offload()
            else:
                pipe.enable_model_cpu_offload()
        elif engine == "flux-schnell":
            from diffusers import FluxPipeline

            pipe = FluxPipeline.from_pretrained(repo, torch_dtype=dtype)
            pipe.enable_model_cpu_offload()
        else:
            raise ValueError(f"Unknown engine {engine}")
        self.pipe, self.pipe_key = pipe, key
        log(f"Loaded in {time.time() - t0:.0f}s")

    # ---------- generation ----------
    def run_video(self, job: dict, dtype) -> dict:
        """One clip. Wan 2.2 shots longer than 5 s are made as several segments of up to
        5 s, each starting from the last frame of the one before, joined into one file."""
        from diffusers.utils import export_to_video

        engine = job["engine"]
        spec = ENGINES[engine]
        fps = spec["fps"]
        mult = spec["mult"]
        width = snap(job.get("width", 1280), mult, 256, 1280)
        height = snap(job.get("height", 704), mult, 256, 1280)
        plan = segment_plan(float(job.get("seconds", 5)), engine)
        seed = job.get("seed")
        seed = int(seed) if seed is not None else random.randint(0, 2**31 - 1)
        start_img = decode_image(job["startImage"]) if job.get("startImage") and engine.startswith("wan2.2") else None
        first_mode = "i2v" if start_img is not None else "t2v"
        joined = []
        total_frames = 0
        for k, seconds in enumerate(plan):
            frames = wan_frames(seconds, fps)
            mode = "i2v" if start_img is not None else "t2v"
            if len(plan) > 1:
                log(f"  segment {k + 1}/{len(plan)}: {seconds:.1f}s ({mode})")
            out = self._wan_segment(job, dtype, engine, mode, width, height, frames, seed + k, start_img)
            if frames_look_broken(out):
                raise FloatingPointError("output frames are blank (precision problem)")
            seg = [to_uint8(f) for f in out]
            del out
            # a continuation starts on the previous segment's last frame: drop the duplicate
            joined.extend(seg if k == 0 else seg[1:])
            total_frames = len(joined)
            start_img = to_pil(seg[-1])
        name = f"{job['id']}.mp4"
        path = os.path.join(OUT_DIR, name)
        # export_to_video multiplies numpy frames by 255, so hand it images
        export_to_video([to_pil(f) for f in joined], path, fps=fps)
        del joined
        mux_ok = reencode_h264(path)
        return {"file": name, "width": width, "height": height, "frames": total_frames, "fps": fps,
                "seed": seed, "mode": first_mode, "segments": len(plan), "h264": mux_ok}

    def _wan_segment(self, job, dtype, engine, mode, width, height, frames, seed, image):
        self.load(engine, mode, dtype)
        gen = self.torch.Generator(device="cpu").manual_seed(seed)
        prompts = dict(prompt=job["prompt"], negative_prompt=job.get("negative") or NEGATIVE_VIDEO)
        if self.split:
            self.pipe.text_encoder = self.te
            with self.torch.no_grad():
                pe, npe = self.pipe.encode_prompt(
                    prompt=prompts["prompt"], negative_prompt=prompts["negative_prompt"],
                    do_classifier_free_guidance=True, device=self.torch.device("cuda:1"), dtype=dtype)
            self.pipe.text_encoder = None
            prompts = dict(prompt_embeds=pe.to("cuda:0"), negative_prompt_embeds=npe.to("cuda:0"))
        kwargs = dict(
            **prompts,
            height=height,
            width=width,
            num_frames=frames,
            guidance_scale=float(job.get("guidance", 5.0)),
            num_inference_steps=int(job.get("steps") or 30),
            generator=gen,
        )
        if mode == "i2v":
            kwargs["image"] = fit_cover(image, width, height)
        return self.pipe(**kwargs).frames[0]

    def run_image(self, job: dict, dtype) -> dict:
        engine = job["engine"]
        mult = ENGINES[engine]["mult"]
        width = snap(job.get("width", 1280), mult, 256, 2048)
        height = snap(job.get("height", 720), mult, 256, 2048)
        seed = job.get("seed")
        seed = int(seed) if seed is not None else random.randint(0, 2**31 - 1)
        self.load(engine, "t2i", dtype)
        gen = self.torch.Generator(device="cpu").manual_seed(seed)
        if engine == "z-image-turbo":
            img = self.pipe(prompt=job["prompt"], height=height, width=width,
                            num_inference_steps=int(job.get("steps") or 9), guidance_scale=0.0,
                            generator=gen).images[0]
        else:
            img = self.pipe(prompt=job["prompt"], height=height, width=width,
                            num_inference_steps=int(job.get("steps") or 4), guidance_scale=0.0,
                            max_sequence_length=256, generator=gen).images[0]
        if frames_look_broken(img):
            raise FloatingPointError("output image is blank (precision problem)")
        name = f"{job['id']}.png"
        img.save(os.path.join(OUT_DIR, name))
        return {"file": name, "width": width, "height": height, "seed": seed}

    def run(self, job: dict) -> dict:
        kind = ENGINES[job["engine"]]["kind"]
        last_err = None
        for dtype in self.dtypes_for(job["engine"]):
            t0 = time.time()
            try:
                res = self.run_video(job, dtype) if kind == "video" else self.run_image(job, dtype)
                res.update(ok=True, seconds_taken=round(time.time() - t0, 1), dtype=str(dtype).replace("torch.", ""))
                return res
            except FloatingPointError as e:
                last_err = str(e)
                e.__traceback__ = None  # drop frame references so the old tensors can be freed
                log(f"  {e}; retrying with the next precision")
                self.unload()
            except Exception as e:  # noqa: BLE001
                self.unload()
                return {"ok": False, "error": f"{type(e).__name__}: {e}", "trace": traceback.format_exc()[-1500:],
                        "seconds_taken": round(time.time() - t0, 1)}
        return {"ok": False, "error": last_err or "no precision worked"}



MAX_SEGMENT_S = 5.0
MAX_LONG_S = 20.0


def segment_plan(seconds: float, engine: str) -> list[float]:
    """Split a requested length into equal segments of at most 5 s. Only Wan 2.2 can continue
    from a frame, so other engines stay a single clip of at most 5 s."""
    if not engine.startswith("wan2.2"):
        return [max(1.0, min(seconds, MAX_SEGMENT_S))]
    total = max(1.0, min(seconds, MAX_LONG_S))
    n = max(1, math.ceil(total / MAX_SEGMENT_S - 1e-9))
    return [round(total / n, 3)] * n


def wan_frames(seconds: float, fps: int) -> int:
    frames = int(round(seconds * fps))
    frames = ((frames - 1) // 4) * 4 + 1  # Wan needs 4k+1 frames
    return max(17, frames)


def to_uint8(frame):
    import numpy as np

    a = np.asarray(frame)
    if a.dtype != np.uint8:
        a = (np.clip(a, 0.0, 1.0) * 255.0 + 0.5).astype(np.uint8)
    return a


def to_pil(frame):
    from PIL import Image

    return Image.fromarray(to_uint8(frame))


def reencode_h264(path: str) -> bool:
    """Re-encode to H.264 + yuv420p so browsers and Kathaa play it everywhere."""
    tmp = path + ".tmp.mp4"
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-c:v", "libx264", "-preset", "medium",
           "-crf", "16", "-pix_fmt", "yuv420p", "-movflags", "+faststart", tmp]
    try:
        subprocess.run(cmd, check=True)
        os.replace(tmp, path)
        return True
    except Exception:  # noqa: BLE001
        if os.path.exists(tmp):
            os.remove(tmp)
        return False


def validate(spec: dict) -> list[dict]:
    if not isinstance(spec, dict) or spec.get("app") != "kathaa" or not isinstance(spec.get("jobs"), list):
        raise ValueError("JOBS must be the JSON exported by Kathaa (app = 'kathaa', with a jobs list).")
    jobs = []
    for j in spec["jobs"]:
        if j.get("engine") not in ENGINES:
            raise ValueError(f"Unknown engine {j.get('engine')!r} in job {j.get('id')}")
        if not str(j.get("prompt", "")).strip():
            raise ValueError(f"Job {j.get('id')} has no prompt")
        jobs.append(j)
    return jobs


def main(spec: dict) -> dict:
    os.makedirs(OUT_DIR, exist_ok=True)
    jobs = validate(spec)
    env = gpu_summary()
    log(f"GPU: {env['gpus']}  torch {env['torch']}  diffusers {env['diffusers']}")
    manifest = {"app": "kathaa", "type": "results", "version": 1, "worker": "kaggle", "env": env,
                "started": now_iso(), "results": []}
    # Free disk held by models this batch does not need (Kaggle keeps about 57 GB per session).
    needed = {j["engine"] for j in jobs}
    for name, spec_ in ENGINES.items():
        if name not in needed:
            purge_model_cache(spec_["repo"])
    worker = Worker()
    # Group by engine so each model loads once.
    order = sorted(range(len(jobs)), key=lambda i: (jobs[i]["engine"], bool(jobs[i].get("startImage")), i))
    previous = None
    for n, i in enumerate(order, 1):
        job = jobs[i]
        if previous and previous != job["engine"]:
            worker.unload()
            purge_model_cache(ENGINES[previous]["repo"])
        previous = job["engine"]
        log(f"Free disk: {free_disk_gb()} GB")
        log(f"Job {n}/{len(jobs)}  {job['id']}  {job['engine']}  “{job.get('title') or job['prompt'][:50]}”")
        res = worker.run(job)
        res.update(id=job["id"], engine=job["engine"], kind=ENGINES[job["engine"]]["kind"],
                   prompt=job["prompt"], title=job.get("title", ""), finished=now_iso())
        manifest["results"].append(res)
        log(("  done" if res["ok"] else "  FAILED: " + res.get("error", "")) + f" in {res.get('seconds_taken', 0)}s")
        with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
            json.dump(manifest, f, indent=2)
    manifest["finished"] = now_iso()
    with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    ok = sum(r["ok"] for r in manifest["results"])
    log(f"Finished: {ok}/{len(jobs)} succeeded. Files are in {OUT_DIR}")
    return manifest




In [ ]:
# 4 — Run the batch
import json
manifest = main(json.loads(JOBS))
for r in manifest['results']:
    print(('OK  ' if r['ok'] else 'FAIL'), r['id'], r.get('seconds_taken'), 's', r.get('file') or r.get('error'))

In [ ]:
# 5 — One zip for Kathaa (Output tab → kathaa-results.zip)
import shutil
print(shutil.make_archive('/kaggle/working/kathaa-results', 'zip', OUT_DIR))